# Export LiDAR tiles as compact f16 grids

Re-exports each original 1 km × 1 km Vancouver LiDAR tile as a raw stream of float16 values plus one shared `index.csv`.

**Per tile file `<NAME>.f16`**: `width * height` little-endian IEEE float16 values, no header.
- Row-major and north-up: value `k` is at `row = k // width` and `col = k % width`. Row 0 is the northern edge and col 0 is the western edge.
- Each value is the maximum altitude in the cell, normalised to `[-1, 1]` over the tile's `[alt_min, alt_max]`:
  `alt = alt_min + (v + 1) / 2 * (alt_max - alt_min)` (metres). The error is at most `(alt_max - alt_min) / 8192`.
- `NaN` means the cell has no points.

**`index.csv`**: one row per tile.
- `x_start, y_start` / `x_end, y_end`: UTM (`crs`, EPSG:26910) centres of the first cell (NW, k = 0) and the last cell (SE, k = width*height-1).
  Cell centre: `x = x_start + col * (x_end - x_start) / (width - 1)`, `y = y_start + row * (y_end - y_start) / (height - 1)`.
- `lon_*, lat_*`: WGS84 coordinates of the four corner cell centres (`nw` = start, `se` = end). Bilinear interpolation between them gives lat/lon for any cell, accurate to about 2e-7° (≈ 2.5 cm, far below the 0.5 m cell size), so no projection library is needed.
- `alt_min, alt_max`: the range used for normalisation. `n_points` and `n_missing` are the points used and the empty cells.

Grids are aligned on the tile's nominal bounds (taken from its name, the SW corner), so neighbouring tiles join seamlessly even when a coastal tile has sparse data.

In [ ]:
import zipfile
import pathlib
import laspy
import numpy as np
import pandas as pd
from pyproj import Transformer

base_path = pathlib.Path("~/").expanduser()
data_path = base_path / "data"
lidar_path = data_path / "raw" / "lidar-vancouver"
temp_path = base_path / "temp"
export_path = data_path / "lidar-vancouver-f16"

index_df = pd.read_csv(lidar_path / "index.csv", sep=";")

tile_size = 1000  # metres; tile NAME is the SW corner "<x>_<y>" in EPSG:26910
resolution = 2000  # cells per side -> 0.5 m cells, 8 MB per tile
noise_classes = [7, 18]  # ASPRS low / high noise: would otherwise dominate the per-cell max

In [ ]:
def load_lidar(name):
    las_file = temp_path / f"{name}.las"
    try:
        # The uncompressed file does not need to live long
        with zipfile.ZipFile(lidar_path / f"{name}.zip", 'r') as zip_ref:
            zip_ref.extract(las_file.name, temp_path)
        las = laspy.read(las_file)
        keep = ~np.isin(np.asarray(las.classification), noise_classes)
        return np.asarray(las.x)[keep], np.asarray(las.y)[keep], np.asarray(las.z)[keep], las.header.parse_crs()
    finally:
        las_file.unlink(missing_ok=True)


def grid_tile(name):
    """Max altitude per cell on a north-up (row 0 = north) resolution x resolution grid. NaN = no points."""
    x, y, z, crs = load_lidar(name)
    x0, y0 = (int(v) for v in name.split("_"))
    cell = tile_size / resolution
    col = np.clip(np.floor((x - x0) / cell).astype(np.int64), 0, resolution - 1)
    row = np.clip(np.floor((y0 + tile_size - y) / cell).astype(np.int64), 0, resolution - 1)
    grid = np.full(resolution * resolution, -np.inf)
    np.maximum.at(grid, row * resolution + col, z)
    grid[np.isneginf(grid)] = np.nan
    return grid.reshape(resolution, resolution), crs, len(z)


def tile_metadata(name, grid, crs, n_points):
    x0, y0 = (int(v) for v in name.split("_"))
    half = tile_size / resolution / 2
    # Centres of the first (NW) and last (SE) cells
    x_start, y_start = x0 + half, y0 + tile_size - half
    x_end, y_end = x0 + tile_size - half, y0 + half
    corners = {"nw": (x_start, y_start), "ne": (x_end, y_start), "sw": (x_start, y_end), "se": (x_end, y_end)}
    transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
    meta = {
        "name": name, "file": f"{name}.f16", "crs": f"EPSG:{crs.to_epsg()}",
        "width": resolution, "height": resolution,
        "x_start": x_start, "y_start": y_start, "x_end": x_end, "y_end": y_end,
    }
    for key, (cx, cy) in corners.items():
        meta[f"lon_{key}"], meta[f"lat_{key}"] = transformer.transform(cx, cy)
    has_data = not np.isnan(grid).all()
    meta["alt_min"] = float(np.nanmin(grid)) if has_data else 0.0
    meta["alt_max"] = float(np.nanmax(grid)) if has_data else 0.0
    meta["n_points"] = n_points
    meta["n_missing"] = int(np.isnan(grid).sum())
    return meta


def encode(grid, alt_min, alt_max):
    """Normalise to [-1, 1] (uses the f16 sign bit, halving the error vs [0, 1]). NaN stays NaN."""
    span = alt_max - alt_min
    norm = (grid - alt_min) / span * 2 - 1 if span > 0 else np.zeros_like(grid)
    return norm.astype("<f2")


def decode(stream, alt_min, alt_max):
    return alt_min + (stream.astype(np.float64) + 1) / 2 * (alt_max - alt_min)

In [ ]:
# Try one tile and check the round-trip error
name = index_df["NAME"][0]
grid, crs, n_points = grid_tile(name)
meta = tile_metadata(name, grid, crs, n_points)
decoded = decode(encode(grid, meta["alt_min"], meta["alt_max"]), meta["alt_min"], meta["alt_max"])
err = np.nanmax(np.abs(decoded - grid))
print(f"max abs error {err:.4f} m (bound {(meta['alt_max'] - meta['alt_min']) / 8192:.4f} m)")
pd.Series(meta)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(decoded, cmap="terrain")  # imshow is north-up for row 0 = north
plt.colorbar(label="altitude (m)")

In [ ]:
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

export_all = True
# Each worker peaks at ~5 GB of RAM on the densest tiles; 12 workers got OOM-killed on the 52 GB server
max_workers = 8

def export_tile(name):
    grid, crs, n_points = grid_tile(name)
    meta = tile_metadata(name, grid, crs, n_points)
    out_file = export_path / meta["file"]
    # Write to a temp file first so an interrupted run never leaves a partial tile behind
    tmp_file = out_file.with_suffix(".f16.tmp")
    encode(grid, meta["alt_min"], meta["alt_max"]).tofile(tmp_file)
    tmp_file.replace(out_file)
    return meta

if export_all:
    export_path.mkdir(parents=True, exist_ok=True)
    temp_path.mkdir(parents=True, exist_ok=True)
    rows, failed = [], {}
    # fork so workers can see the functions defined in this notebook
    with ProcessPoolExecutor(max_workers, mp_context=multiprocessing.get_context("fork")) as pool:
        futures = {pool.submit(export_tile, name): name for name in index_df["NAME"]}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Exporting tiles"):
            try:
                rows.append(future.result())
            except Exception as e:
                failed[futures[future]] = repr(e)
    out_index = pd.DataFrame(rows).sort_values("name")
    out_index.to_csv(export_path / "index.csv", index=False, float_format="%.12g")
    print(f"Done: {len(rows)} ok, {len(failed)} failed")
    for name, err in failed.items():
        print(f"  {name}: {err}")

In [ ]:
# Read back exactly as an external program would: only index.csv + the raw stream
index = pd.read_csv(export_path / "index.csv")
t = index.iloc[0]
v = np.fromfile(export_path / t["file"], dtype="<f2")
alt = t["alt_min"] + (v.astype(np.float64) + 1) / 2 * (t["alt_max"] - t["alt_min"])
k = np.arange(v.size)
row, col = k // t["width"], k % t["width"]
x = t["x_start"] + col * (t["x_end"] - t["x_start"]) / (t["width"] - 1)
y = t["y_start"] + row * (t["y_end"] - t["y_start"]) / (t["height"] - 1)
# Bilinear lat/lon from the corner cells
u, w = col / (t["width"] - 1), row / (t["height"] - 1)
bilerp = lambda c: (1 - w) * ((1 - u) * t[f"{c}_nw"] + u * t[f"{c}_ne"]) + w * ((1 - u) * t[f"{c}_sw"] + u * t[f"{c}_se"])
lon, lat = bilerp("lon"), bilerp("lat")

# Compare against an exact projection
exact_lon, exact_lat = Transformer.from_crs(t["crs"], "EPSG:4326", always_xy=True).transform(x, y)
print("max lat/lon error (deg):", np.abs(lat - exact_lat).max(), np.abs(lon - exact_lon).max())
print("total size:", sum(f.stat().st_size for f in export_path.iterdir()) / 2**20, "MiB")
pd.DataFrame({"x": x, "y": y, "lat": lat, "lon": lon, "alt": alt}).dropna()

## u16 export with a global altitude range

Same layout and `index.csv` columns as the f16 export, but in `data/lidar-vancouver-u16/` with `<NAME>.u16` files. Each file is `width * height` little-endian uint16 values.
- `alt_min, alt_max` are the **same for every tile**: the global range of the whole dataset.
- `alt = alt_min + v / 65534 * (alt_max - alt_min)`. `v = 65535` means the cell has no points.
- The step is `(alt_max - alt_min) / 65534`. For -10.8 to 588.4 m that is 9.1 mm, so the error is at most 4.6 mm.

The global range comes from the per-tile ranges in the f16 `index.csv` (same gridding and noise filtering), so the f16 export must run first.

In [ ]:
u16_path = data_path / "lidar-vancouver-u16"
u16_missing = 65535

f16_index = pd.read_csv(export_path / "index.csv")
assert len(f16_index) == len(index_df), "run the f16 export first"
global_min, global_max = f16_index["alt_min"].min(), f16_index["alt_max"].max()
print(f"global range {global_min} .. {global_max} m, max error {(global_max - global_min) / 65534 / 2 * 1000:.2f} mm")

def encode_u16(grid):
    v = np.round((grid - global_min) / (global_max - global_min) * (u16_missing - 1))
    return np.where(np.isnan(grid), u16_missing, v).astype("<u2")

def export_tile_u16(name):
    out_file = u16_path / f"{name}.u16"
    if out_file.exists():
        return
    grid, _, _ = grid_tile(name)
    tmp_file = out_file.with_suffix(".u16.tmp")
    encode_u16(grid).tofile(tmp_file)
    tmp_file.replace(out_file)

if export_all:
    u16_path.mkdir(parents=True, exist_ok=True)
    temp_path.mkdir(parents=True, exist_ok=True)
    failed = {}
    with ProcessPoolExecutor(max_workers, mp_context=multiprocessing.get_context("fork")) as pool:
        futures = {pool.submit(export_tile_u16, name): name for name in index_df["NAME"]}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Exporting u16 tiles"):
            try:
                future.result()
            except Exception as e:
                failed[futures[future]] = repr(e)
    # Same grids as the f16 export, so its metadata carries over; only the file and value range change
    u16_index = f16_index.assign(file=f16_index["name"] + ".u16", alt_min=global_min, alt_max=global_max)
    u16_index.to_csv(u16_path / "index.csv", index=False, float_format="%.12g")
    print(f"Done: {len(index_df) - len(failed)} ok, {len(failed)} failed")
    for name, err in failed.items():
        print(f"  {name}: {err}")